# Case Study 4: E-Commerce Customer Segmentation
## RFM Analysis with K-Means, DBSCAN and PCA

**Dataset:** `online_retail_clean.csv` from Mathew K Analytics.

This notebook follows the structure of the reference notebook you shared: dataset inspection, preprocessing checks, missing-value and duplicate checks, IQR outlier detection, RFM creation, RFM visualizations, scaling, K-Means (including the WCSS/elbow plot), DBSCAN, cluster summaries and visualizations. PCA is included to display the clustering results in two dimensions.

**Scope:** RFM, data preprocessing, K-Means, DBSCAN and PCA only. The customer ID is retained for reference but is not used as a clustering feature.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

%matplotlib inline

## 2. Load the dataset

Keep `online_retail_clean.csv` in the same folder as this notebook. If your downloaded file has a different name, update `file_path`.

In [ ]:
file_path = "online_retail_clean.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head(20))

## 3. Inspect the dataset structure

In [ ]:
df.info()

## 4. Check invoice frequency

In [ ]:
df["InvoiceNo"].value_counts().head(10)

## 5. Check missing values

In [ ]:
df.isna().sum()

## 6. Reset the index

In [ ]:
df = df.reset_index(drop=True)
df.head()

## 7. Check duplicate rows

In [ ]:
duplicates = df[df.duplicated()]
print("Number of exact duplicate rows:", len(duplicates))
display(duplicates.head(10))

## 8. Descriptive statistics

In [ ]:
df.describe(include="all").T

## 9. Inspect categorical columns

In [ ]:
for column in df.select_dtypes(include=["object"]).columns:
    print(f"\nColumn: {column}")
    print("Unique values:", df[column].nunique(dropna=True))
    print("Example values:", df[column].dropna().unique()[:10])

## 10. Select numeric transaction columns for outlier detection

`CustomerID` and `InvoiceNo` are identifiers, so they are not treated as numeric behavior measurements. IQR outlier detection is applied to `Quantity`, `UnitPrice` and `Revenue`.

In [ ]:
numeric_columns = ["Quantity", "UnitPrice", "Revenue"]
num_df = df[numeric_columns].copy()
num_df.describe().T

## 11. Detect outliers using the IQR method

In [ ]:
def detect_outliers(column):
    q1 = column.quantile(0.25)
    q3 = column.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return column[(column < lower_bound) | (column > upper_bound)]

outliers_dict = {}
for column in num_df.columns:
    outliers_dict[column] = detect_outliers(num_df[column])

outlier_summary = pd.DataFrame({
    "Column": list(outliers_dict.keys()),
    "Outlier_Count": [len(values) for values in outliers_dict.values()]
})
display(outlier_summary)

for column, outliers in outliers_dict.items():
    if not outliers.empty:
        print(f"\n{column}: {len(outliers)} detected outliers")
        print(outliers.head(10).to_list())

## 12. Visualize the number of outliers

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(outlier_summary["Column"], outlier_summary["Outlier_Count"])
plt.title("Number of IQR Outliers in Transaction Columns")
plt.xlabel("Transaction Feature")
plt.ylabel("Number of Outliers")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 13. Preprocess the transaction data

The file is already a cleaned retail dataset, but these checks make the notebook reproducible: convert dates and numeric fields, remove exact duplicate rows, remove records without a usable customer/date/invoice, exclude cancelled invoices, and keep only positive-quantity, positive-price purchases. The IQR outliers above are **reported but not automatically deleted or capped**; unusually large purchases can be genuine customer behavior.

In [ ]:
clean_df = df.copy()

# Standardize data types
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype("string").str.strip()
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["CustomerID"] = pd.to_numeric(clean_df["CustomerID"], errors="coerce")
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")

if "Revenue" in clean_df.columns:
    clean_df["Revenue"] = pd.to_numeric(clean_df["Revenue"], errors="coerce")
else:
    clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]

row_counts = [{"Step": "Rows loaded", "Rows_Remaining": len(clean_df)}]

# Remove exact duplicate rows
clean_df = clean_df.drop_duplicates()
row_counts.append({"Step": "After removing exact duplicates", "Rows_Remaining": len(clean_df)})

# Remove rows that cannot be assigned to a customer or valid invoice date
clean_df = clean_df.dropna(subset=["CustomerID", "InvoiceDate", "InvoiceNo", "Quantity", "UnitPrice", "Revenue"])
row_counts.append({"Step": "After removing missing required values", "Rows_Remaining": len(clean_df)})

# Remove cancelled invoices (invoice numbers beginning with C)
clean_df = clean_df[~clean_df["InvoiceNo"].str.upper().str.startswith("C", na=False)]
row_counts.append({"Step": "After removing cancelled invoices", "Rows_Remaining": len(clean_df)})

# Keep valid purchase rows only
clean_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["UnitPrice"] > 0) & (clean_df["Revenue"] > 0)]
row_counts.append({"Step": "After keeping positive purchases", "Rows_Remaining": len(clean_df)})

clean_df["CustomerID"] = clean_df["CustomerID"].astype("int64")
clean_df = clean_df.reset_index(drop=True)

row_counts_df = pd.DataFrame(row_counts)
row_counts_df["Rows_Removed_From_Previous_Step"] = row_counts_df["Rows_Remaining"].shift(1) - row_counts_df["Rows_Remaining"]
row_counts_df.loc[0, "Rows_Removed_From_Previous_Step"] = 0
row_counts_df["Rows_Removed_From_Previous_Step"] = row_counts_df["Rows_Removed_From_Previous_Step"].fillna(0).astype(int)
display(row_counts_df)
print("Final cleaned transaction shape:", clean_df.shape)
display(clean_df.head())

## 14. Verify the preprocessed data

In [ ]:
print("Missing values after preprocessing:")
display(clean_df.isna().sum().to_frame("Missing values"))
print("Exact duplicate rows after preprocessing:", clean_df.duplicated().sum())
print("Unique customers:", clean_df["CustomerID"].nunique())
print("Unique invoices:", clean_df["InvoiceNo"].nunique())

## 15. Create the RFM table

- **Recency:** days since the customer's most recent purchase, measured relative to one day after the latest date in the cleaned dataset.
- **Frequency:** number of distinct invoices for the customer.
- **Monetary:** sum of `Revenue` for the customer.

In [ ]:
snapshot_date = clean_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_table = clean_df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda dates: (snapshot_date - dates.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

print("RFM table shape:", rfm_table.shape)
display(rfm_table.head(20))

## 16. Clean and summarize the RFM table

In [ ]:
rfm_table_cleaned = rfm_table.dropna(subset=["Recency", "Frequency", "Monetary"]).copy()
rfm_table_cleaned = rfm_table_cleaned[
    (rfm_table_cleaned["Recency"] >= 0)
    & (rfm_table_cleaned["Frequency"] > 0)
    & (rfm_table_cleaned["Monetary"] > 0)
].reset_index(drop=True)

print("Customers available for clustering:", len(rfm_table_cleaned))
display(rfm_table_cleaned[["Recency", "Frequency", "Monetary"]].describe().T)

## 17. Visualization: Recency vs Monetary

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(rfm_table_cleaned["Recency"], rfm_table_cleaned["Monetary"], alpha=0.65)
plt.yscale("log")
plt.title("Recency vs Monetary")
plt.xlabel("Recency (days)")
plt.ylabel("Monetary Value (log scale)")
plt.tight_layout()
plt.show()

## 18. Visualization: Frequency vs Monetary

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(rfm_table_cleaned["Frequency"], rfm_table_cleaned["Monetary"], alpha=0.65)
plt.yscale("log")
plt.title("Frequency vs Monetary")
plt.xlabel("Frequency (number of invoices)")
plt.ylabel("Monetary Value (log scale)")
plt.tight_layout()
plt.show()

## 19. Prepare the RFM features for clustering

Only `Recency`, `Frequency` and `Monetary` are used as model inputs. `CustomerID` is kept separately to identify each customer and is not included in the distance calculations.

In [ ]:
data = rfm_table_cleaned[["CustomerID", "Recency", "Frequency", "Monetary"]].copy()
rfm_features = data[["Recency", "Frequency", "Monetary"]]

sc = StandardScaler()
data_scaled = sc.fit_transform(rfm_features)

print("Scaled feature shape:", data_scaled.shape)
print("Feature names:", rfm_features.columns.tolist())

## 20. K-Means: WCSS / Elbow method

In [ ]:
wcss = []
cluster_range = range(2, 10)

for k in cluster_range:
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_model.fit(data_scaled)
    wcss.append(kmeans_model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(cluster_range), wcss, marker="o")
plt.title("Number of Clusters vs WCSS (Elbow Method)")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.xticks(list(cluster_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 21. Apply K-Means with 4 clusters

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_pred_kmeans = kmeans.fit_predict(data_scaled)

data_kmeans = data.copy()
data_kmeans["KMeans_Cluster"] = y_pred_kmeans

print("K-Means cluster labels:", np.unique(y_pred_kmeans))
print("Customers in each K-Means cluster:")
display(data_kmeans["KMeans_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))

print("RFM summary by K-Means cluster:")
display(data_kmeans.groupby("KMeans_Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(2))

## 22. Visualization: K-Means cluster sizes

In [ ]:
kmeans_counts = data_kmeans["KMeans_Cluster"].value_counts().sort_index().rename_axis("KMeans_Cluster").reset_index(name="Customer_Count")
plt.figure(figsize=(8, 6))
plt.bar(kmeans_counts["KMeans_Cluster"].astype(str), kmeans_counts["Customer_Count"])
plt.title("Number of Customers in Each K-Means Cluster")
plt.xlabel("KMeans_Cluster")
plt.ylabel("Customer_Count")
plt.tight_layout()
plt.show()

## 23. Apply DBSCAN (eps = 0.5, min_samples = 5)

In [ ]:
db = DBSCAN(eps=0.5, min_samples=5)
y_pred_db = db.fit_predict(data_scaled)

data_db_05 = data.copy()
data_db_05["DBSCAN_Cluster"] = y_pred_db

print("DBSCAN labels (including -1 for noise):", np.unique(y_pred_db))
print("Customers in each DBSCAN label:")
display(data_db_05["DBSCAN_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))

print("RFM descriptive statistics by DBSCAN label:")
display(data_db_05.groupby("DBSCAN_Cluster")[["Recency", "Frequency", "Monetary"]].describe().T)

## 24. Apply DBSCAN again (eps = 0.4, min_samples = 5)

In [ ]:
db_04 = DBSCAN(eps=0.4, min_samples=5)
y_pred_db_04 = db_04.fit_predict(data_scaled)

data_db_04 = data.copy()
data_db_04["DBSCAN_Cluster"] = y_pred_db_04

print("DBSCAN labels for eps=0.4 (including -1 for noise):", np.unique(y_pred_db_04))
display(data_db_04["DBSCAN_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))
print("RFM descriptive statistics by DBSCAN label (eps=0.4):")
display(data_db_04.groupby("DBSCAN_Cluster")[["Recency", "Frequency", "Monetary"]].describe().T)

## 25. Reduce dimensions with PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(data_scaled)

pca_data = data.copy()
pca_data["PCA1"] = rfm_pca[:, 0]
pca_data["PCA2"] = rfm_pca[:, 1]
pca_data["KMeans_Cluster"] = y_pred_kmeans
pca_data["DBSCAN_eps_05"] = y_pred_db
pca_data["DBSCAN_eps_04"] = y_pred_db_04

print("Explained variance ratio for the two principal components:", pca.explained_variance_ratio_)
print("Total variance represented in 2D:", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")
display(pca_data.head())

## 26. PCA visualization: K-Means clusters

In [ ]:
plt.figure(figsize=(8, 6))
for cluster in pca_data["KMeans_Cluster"].astype(str).unique():
    mask = pca_data["KMeans_Cluster"].astype(str) == cluster
    plt.scatter(pca_data.loc[mask, "PCA1"], pca_data.loc[mask, "PCA2"], label=cluster, alpha=0.7)
plt.title("K-Means Clusters Visualized with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="K-Means Cluster")
plt.tight_layout()
plt.show()

## 27. PCA visualization: DBSCAN (eps = 0.5)

In [ ]:
plt.figure(figsize=(8, 6))
for cluster in pca_data["DBSCAN_eps_05"].astype(str).unique():
    mask = pca_data["DBSCAN_eps_05"].astype(str) == cluster
    plt.scatter(pca_data.loc[mask, "PCA1"], pca_data.loc[mask, "PCA2"], label=cluster, alpha=0.7)
plt.title("DBSCAN Clusters Visualized with PCA (eps=0.5)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="DBSCAN Label (-1 = Noise)")
plt.tight_layout()
plt.show()

## 28. PCA visualization: DBSCAN (eps = 0.4)

In [ ]:
plt.figure(figsize=(8, 6))
for cluster in pca_data["DBSCAN_eps_04"].astype(str).unique():
    mask = pca_data["DBSCAN_eps_04"].astype(str) == cluster
    plt.scatter(pca_data.loc[mask, "PCA1"], pca_data.loc[mask, "PCA2"], label=cluster, alpha=0.7)
plt.title("DBSCAN Clusters Visualized with PCA (eps=0.4)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="DBSCAN Label (-1 = Noise)")
plt.tight_layout()
plt.show()

## 29. Compare K-Means and DBSCAN cluster counts

In [ ]:
print("K-Means cluster counts:")
display(data_kmeans["KMeans_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))

print("DBSCAN cluster counts, eps=0.5 (-1 = noise):")
display(data_db_05["DBSCAN_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))

print("DBSCAN cluster counts, eps=0.4 (-1 = noise):")
display(data_db_04["DBSCAN_Cluster"].value_counts().sort_index().to_frame("Customer_Count"))

## 30. Save the final customer-level results

The output CSV includes each customer's RFM values, K-Means label, both DBSCAN labels, and the PCA coordinates used in the visualizations.

In [ ]:
final_results = pca_data.copy()
final_results.to_csv("ecommerce_customer_segmentation_results.csv", index=False)
print("Saved ecommerce_customer_segmentation_results.csv")
display(final_results.head(10))

## Conclusion

The transaction data was inspected and preprocessed before it was aggregated into one RFM row per customer. K-Means and DBSCAN were applied to the **scaled RFM features**. PCA reduced the three RFM dimensions to two principal components for cluster visualization. DBSCAN label `-1` represents customers classified as noise.

PCA is used for visualization only; the clustering algorithms use the three scaled RFM features. Cluster counts and shapes depend on the data and the DBSCAN `eps` setting.